In [1]:
# %pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

In [ ]:
import pixeltable_pgserver as pgserver # аналог pgserver
from sqlalchemy import create_engine, text
import pandas as pd
import os
import kagglehub

db = pgserver.get_server("./temp", cleanup_mode="delete")

uri = db.get_uri(driver="psycopg2")
print("Connection URI:", uri)
print(db.get_postmaster_info().pgdata)
print(db.get_postmaster_info().port)

engine = create_engine(uri, echo=False, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

Connection URI: postgresql+psycopg2://postgres:@/postgres?host=/Users/vladyslavratushnyi/Learning/DB_DS/hw5/temp
/Users/vladyslavratushnyi/Learning/DB_DS/hw5/temp
5432
PostgreSQL 18.4 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [3]:
# кастомна функція для виконання sql
def sql_exec(sql_quary:str, engine=engine, is_file=False, printout=False)->list:
    try:
        if is_file:
            with open(sql_quary, 'r') as f:
                sql_quary = f.read()

        ddl_statements = [
            query.strip() + ";" 
            for query in sql_quary.split(';') 
            if query.strip()]

        with engine.begin() as conn:
            for stmt in ddl_statements:
                output = conn.execute(text(stmt))
                if output.returns_rows and printout:
                    for i in output.mappings().fetchall():
                        print(i)


        return f"{len(ddl_statements)} команд виконано"

    except Exception as e:
        print('Операція не виконана:', e)


def sql_reader(query:str, engine=engine, is_file=False)->pd.DataFrame:
    try:
        if is_file:
            with open(query, 'r') as f:
                query = f.read()

        with engine.connect() as con:
            return pd.read_sql_query(query, con)

    except Exception as e:
        print('Операція не виконана:', e)

# Setup і typed-схема Olist

In [4]:
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce', output_dir='./data')
print(dataset_dir)

100%|██████████| 42.6M/42.6M [00:01<00:00, 25.9MB/s]

Extracting files...


./data


In [5]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [6]:
sql_exec('sql_comnds/com1.sql', is_file=True)

'10 команд виконано'

In [7]:
sql_exec('sql_comnds/com2.sql', is_file=True)

'5 команд виконано'

In [8]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
"""
sql_reader(query, engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


# CTE-chain pipeline з ≥5 stages

In [9]:
sql_reader('sql_comnds/cte_chain.sql', engine, is_file=True)

,customer_unique_id,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d,cancellation_rate,preferred_category
0,0a0a92112bd4c708ca5fde585afaa872,384,1,13664.08,2017-07-01,13664.080,0.0,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,565,2,7571.63,2017-04-01,3785.815,0.0,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,95,1,7274.88,2018-07-01,7274.880,0.0,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,613,1,6929.31,2017-01-01,6929.310,0.0,utilidades_domesticas
4,459bef486812aa25204be022145caa62,85,1,6922.21,2018-07-01,6922.210,0.0,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,512,1,6726.66,2017-04-01,6726.660,0.0,artes
6,4007669dec559734d6f53e029e360987,328,1,6081.54,2017-10-01,6081.540,0.0,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,548,1,4764.34,2017-04-01,4764.340,0.0,eletroportateis
8,48e1ac109decbb87765a3eade6854098,118,1,4681.78,2018-04-01,4681.780,0.0,pcs
9,c8460e4251689ba205045f3ea17884a1,71,4,4655.88,2018-07-01,1163.970,0.0,telefonia


### **Логіка по stage**

**Stage 1** збирає по одному рядку на замовлення. Позиції агрегуються в order_total окремо, до приєднання, щоб не розмножити рядки. LEFT JOIN потрібен, бо скасовані й недоступні замовлення часто не мають позицій (тоді order_total = 0).

**Stage 2** рахує віконне середнє. RANGE ... INTERVAL '30 days' PRECEDING бере саме календарні 30 днів, а не 30 рядків (потрібен PostgreSQL 11+). FILTER (WHERE is_valid) виключає скасовані замовлення із середнього.

**Stage 3** згортає все до рівня клієнта. avg_order_value_30d на цьому рівні береться на момент останнього валідного замовлення: ARRAY_AGG(... ORDER BY purchase_ts DESC)[1].

**Stages 4–5** окремо рахують виручку по категоріях і вибирають топ-1 через ROW_NUMBER(). Другий ключ сортування (category) робить результат детермінованим при рівній виручці.

**Stage 6** збирає фінальну таблицю й додає cohort_quarter за вашою формулою.

### **Інші пояснення**

Валідне замовлення — це будь-який статус, окрім canceled і unavailable. Якщо за викладачем потрібні лише delivered, змініть вираз в is_valid, і все далі підлаштується саме.

cancellation_rate рахується по всіх замовленнях клієнта: частка canceled від усіх.

Для клієнтів без жодного валідного замовлення frequency = 0, monetary = 0, а recency_days, avg_order_value_30d і preferred_category будуть NULL.

Товари без категорії потрапляють у 'unknown'.

# Recursive CTE для referral chain

In [10]:
sql_exec('sql_comnds/cte_in_table.sql', is_file=True)

'3 команд виконано'

In [11]:
query = """
SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
"""
sql_reader(query)

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-04 21:11:40.091433+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-04 21:11:40.091433+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-04 21:11:40.091433+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-04 21:11:40.091433+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-04 21:11:40.091433+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-04 21:11:40.091433+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-04 21:11:40.091433+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-04 21:11:40.091433+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-04 21:11:40.091433+00:00


In [12]:
# recursive CTE
sql_reader('sql_comnds/recursive_cte.sql', engine, is_file=True)

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


### Reflection

**Що станеться без захисту.** 

Recursive CTE працює ітераціями: на кожному кроці береться результат попередньої ітерації й приєднується до hw5_referrals. Якщо в таблиці є пари A → B і B → A, то ланцюжок буде таким: A → B → A → B → A → ... Кожна ітерація знову знаходить «нового» referred-клієнта, тому робоча таблиця ніколи не стає порожньою, умова зупинки не спрацьовує, і запит виконується нескінченно. Із UNION ALL дублікати не відкидаються, тож рядків стає дедалі більше, зростають споживання пам'яті та тимчасових файлів, а запит доводиться вбивати вручну або він зупиняється з помилкою через брак ресурсів. Якби замість UNION ALL був UNION, дублікати відсікалися б, але тут у кожному рядку є depth і path, які постійно змінюються, тому рядки щоразу унікальні й UNION від циклу не рятує.

**Захист 1: масив path_ids.** 

У кожному рядку зберігається масив усіх customer_id, які вже пройдені на шляху від кореня. Умова WHERE NOT r.referred_customer_id = ANY(rd.path_ids) не дозволяє перейти до клієнта, який уже є в цьому масиві. У прикладі A → B → A: на шляху [A, B] спроба додати A відхиляється, бо A вже є в масиві. Гілка обривається, і рекурсія завершується природно. Цей захист точний: він зупиняє саме повернення до вже відвіданого вузла, не обрізаючи легітимні глибокі ланцюжки.

**Захист 2: depth guard.** 

Умова rd.depth < 50 обмежує максимальну глибину. Це страховка на випадок, якщо перший захист чогось не врахував (наприклад, помилка в логіці масиву або дуже довгі некоректні дані). Навіть у найгіршому випадку рекурсія зупиниться на 50-му рівні. Недолік у тому, що це грубе обмеження: якщо реальне дерево буде глибшим за 50, його хвіст буде мовчки обрізано, тому значення варто підбирати з урахуванням очікуваних даних.

# Materialized View з CONCURRENT REFRESH

In [13]:
sql_exec('sql_comnds/mv_pipeline.sql', is_file=True)

'3 команд виконано'

In [14]:
# перевірка матеріалізованого подання
query = """
SELECT COUNT(*) AS n_rows
FROM mv_customer_features;

SELECT *
FROM mv_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
"""
sql_reader(query, engine)

,customer_id,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d,cancellation_rate,preferred_category
0,0a0a92112bd4c708ca5fde585afaa872,384,1,13664.08,2017-07-01,13664.080,0.0,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,565,2,7571.63,2017-04-01,3785.815,0.0,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,95,1,7274.88,2018-07-01,7274.880,0.0,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,613,1,6929.31,2017-01-01,6929.310,0.0,utilidades_domesticas
4,459bef486812aa25204be022145caa62,85,1,6922.21,2018-07-01,6922.210,0.0,pcs


In [15]:
# concurrent refresh
sql_exec('REFRESH MATERIALIZED VIEW CONCURRENTLY mv_customer_features;', is_file=False)

'1 команд виконано'

# EXPLAIN ANALYZE + Reflection

In [16]:
# EXPLAIN на повному CTE-pipeline
explain_file = sql_reader('sql_comnds/explain_cte.sql', is_file=True)
explain_file.to_csv('explain_cte.csv', index=False)

In [17]:
# EXPLAIN на lookup за унікальним ідентифікатором із MV
query = """
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_unique_id
    FROM olist_customers
    ORDER BY customer_unique_id
    LIMIT 1
)
SELECT mv.*
FROM mv_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_unique_id = mv.customer_id;
"""
explain_file = sql_reader(query, engine)
explain_file.to_csv('explain_lookup.csv', index=False)

In [18]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT * FROM mv_customer_features
ORDER BY monetary DESC
LIMIT 15;
"""
explain_file = sql_reader(query, engine)
explain_file.to_csv('explain_mv.csv', index=False)

### Reflection

1. Найдорожчий feature. Найдорожчий етап: Stage 4–5 (виручка по категоріях і ROW_NUMBER для preferred_category). У плані це Sort з external merge Disk: 7160kB під GroupAggregate по (клієнт, категорія), далі Incremental Sort і WindowAgg. Перший рядок цієї гілки з'являється лише на ~690 мс із 1319. Причина в блокувальному Sort, який не вмістився в work_mem (тому й Hash Join має Batches: 2). HashAggregate у плані немає: агрегації йдуть як GroupAggregate по відсортованому входу. Другий за вартістю WindowAgg зі Stage 2 над Sort на 6520kB. Разом temp I/O становить ≈ 52 МБ.

2. Чи кешувати через MV. Так. Top-15 із MV виконується за 17.3 мс (Parallel Seq Scan, top-N heapsort, 1445 буферів), а pipeline за 1319 мс і 10 908 буферів, тобто ≈ 76× повільніше. Точковий lookup ще вираженіший: Index Scan using ux_mv_customer_features_customer_id повертає рядок за 0.017 мс і 4 буфери, тоді як pipeline не може обчислити фічі одного клієнта без повного проходу по даних. Refresh я б зробив щодоби або on-demand після завантаження даних. Датасет історичний, as_of_date змінюється лише з новими замовленнями, а щогодинний REFRESH CONCURRENTLY даремно повторював би ті самі ~1.3 с обчислень.

3. Коли краще звичайний VIEW. Звичайний VIEW краще, коли потрібна актуальність у кожному запиті (дашборд поточних статусів замовлень). Він також підходить, коли запит дешевий завдяки індексам, або коли треба лише сховати складність чи обмежити доступ до колонок без копії даних.

4. Termination recursive CTE. Рекурсія завершується, коли recursive part не повертає нових рядків. У циклі A → B → A без захисту цього не станеться. З UNION ALL і змінними depth/path кожен рядок унікальний, тож запит росте до вичерпання ресурсів. Від цього захищають NOT next_id = ANY(path_ids) (гілка обривається при поверненні в A), depth < 50 як страховка та CYCLE у PostgreSQL 14+.

5. Головний trade-off. Головний trade-off — recompute-cost проти freshness. Перерахунок дорогий (1.3 с, spill на диск), storage-cost MV із 96 096 рядків мізерний, а історичним даним реальний час не потрібен. Тому latency 17 мс замість 1319 мс купується майже безкоштовно.

In [19]:
engine.dispose()
db.cleanup()
print("Сервер зупинено.")

Сервер зупинено.
